# Indonesia's Revealed Comparative Advantage in AI Adoption

**Thesis question.** Which sectors of Indonesia's economy have integrated AI into their production technology, and how does Indonesia's adoption profile compare to US-measured AI exposure?

## Methodology

We interpret work-related Claude usage as an intermediate input to production rather than final consumption. Following Acemoglu (2024), AI enters the economy as a task-level input that substitutes for or complements labor. The observed exposure measure of Massenkoff & McCrory (2026), which serves as our US benchmark, captures the share of an occupation's task bundle that has shifted from labor-performed to AI-performed in observed US usage.

We extend this logic cross-nationally using the Revealed Comparative Advantage (RCA) index of Balassa (1965), which infers a country's specialization from observed patterns of economic activity. The RCA has been validated for application beyond trade to patents, scientific publications, and technology indicators (Laursen, 2015), and is methodologically equivalent to the Anthropic AI Usage Index (Appel et al., 2025), which applies the same ratio at the country level.

For each task or occupation category $k$:

$$RCA_{\text{Indonesia}, k} = \frac{\text{Indonesia's share of its Claude usage in category } k}{\text{Global share of Claude usage in category } k}$$

- RCA > 1 → Indonesia's production technology integrates AI more intensively in this domain than the global norm  
- RCA < 1 → Indonesian production in this domain remains more labor-intensive than the global pattern

## References

1. Acemoglu, D. (2024). *The Simple Macroeconomics of AI*. NBER Working Paper No. 32487.  
2. Balassa, B. (1965). Trade Liberalisation and 'Revealed' Comparative Advantage. *The Manchester School*, 33(2), 99–123.  
3. Laursen, K. (2015). Revealed Comparative Advantage and the Alternatives as Measures of International Specialisation. *Eurasian Business Review*, 5, 99–115.  
4. Appel, R., McCrory, P., Tamkin, A., et al. (2025). Anthropic Economic Index Report: Uneven Geographic and Enterprise AI Adoption. arXiv:2511.15080.  
5. Massenkoff, M. & McCrory, P. (2026). Labor Market Impacts of AI. Anthropic Research.

## Data

- **AEI September 2025 release** (Claude.ai geographic data, August 4–11, 2025)
- **Massenkoff & McCrory (2026)** US observed exposure per occupation (`job_exposure.csv`)
- **O\*NET task statements** for task-to-SOC mapping
- **World Bank working-age population** for AUI computation

## 0. Setup

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from pathlib import Path
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

REPO_ID = 'Anthropic/EconomicIndex'
RELEASE = 'release_2025_09_15'
AEI_DIR = Path('./aei_data')
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

INDONESIA = 'ID'
ASEAN = {
    'BN': 'Brunei', 'KH': 'Cambodia', 'ID': 'Indonesia', 'LA': 'Laos',
    'MY': 'Malaysia', 'MM': 'Myanmar', 'PH': 'Philippines', 'SG': 'Singapore',
    'TH': 'Thailand', 'VN': 'Vietnam',
}
ASEAN_PEERS = [c for c in ASEAN if c != INDONESIA]

SOC_MAJOR_LABELS = {
    '11': 'Management', '13': 'Business & Financial', '15': 'Computer & Mathematical',
    '17': 'Architecture & Engineering', '19': 'Life, Physical & Social Science',
    '21': 'Community & Social Service', '23': 'Legal', '25': 'Educational Instruction',
    '27': 'Arts, Design, Entertainment', '29': 'Healthcare Practitioners',
    '31': 'Healthcare Support', '33': 'Protective Service', '35': 'Food Preparation',
    '37': 'Building & Grounds Cleaning', '39': 'Personal Care & Service', '41': 'Sales',
    '43': 'Office & Admin Support', '45': 'Farming, Fishing, Forestry',
    '47': 'Construction & Extraction', '49': 'Installation, Maintenance, Repair',
    '51': 'Production', '53': 'Transportation & Material Moving',
}

print('Setup complete.')

Setup complete.


## 1. Load data and filter

In [5]:
# Download what's needed 
for f in [
    f'{RELEASE}/data/intermediate/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv',
    f'{RELEASE}/data/intermediate/working_age_pop_2024_country.csv',
    f'{RELEASE}/data/intermediate/onet_task_statements.csv',
    'labor_market_impacts/job_exposure.csv',
]:
    try:
        hf_hub_download(repo_id=REPO_ID, filename=f, repo_type='dataset', local_dir=AEI_DIR)
    except Exception as e:
        print(f'Could not fetch {f}: {e}')

geo = pd.read_csv(AEI_DIR / RELEASE / 'data/intermediate/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv')
pop = pd.read_csv(AEI_DIR / RELEASE / 'data/intermediate/working_age_pop_2024_country.csv')
onet_tasks = pd.read_csv(AEI_DIR / RELEASE / 'data/intermediate/onet_task_statements.csv')
job_exposure = pd.read_csv(AEI_DIR / 'labor_market_impacts/job_exposure.csv')

print(f'geo (raw):      {geo.shape}')

# Filter out 'not_classified' from country dim and cluster dim
#   geo_id='not_classified' -> 15.7% of global usage with no country attributed (VPN / missing IP)
#   cluster_name='not_classified' -> conversations Anthropic's classifier could not categorize
n_before = len(geo)
geo = geo[geo.geo_id != 'not_classified'].copy()
geo = geo[geo.cluster_name.fillna('').str.lower() != 'not_classified'].copy()
print(f'geo (clean):    {geo.shape}  (removed {n_before - len(geo):,} not_classified rows)')
print(f'pop:            {pop.shape}')
print(f'onet_tasks:     {onet_tasks.shape}')
print(f'job_exposure:   {job_exposure.shape}')

geo (raw):      (100062, 10)
geo (clean):    (98656, 10)  (removed 1,406 not_classified rows)
pop:            (194, 5)
onet_tasks:     (19530, 9)
job_exposure:   (756, 3)


In [6]:
# What facets are available?
print("Facets in geo data:")
print(geo.facet.value_counts())

# Are there purpose/use-case variables?
print("\nVariables per facet:")
for facet in geo.facet.unique():
    vars_in_facet = geo[geo.facet == facet].variable.unique()
    print(f"  {facet}: {list(vars_in_facet)}")

Facets in geo data:
facet
request                     49464
onet_task                   23756
onet_task::collaboration    14454
request::collaboration       8508
collaboration                2028
country                       344
state_us                      102
Name: count, dtype: int64

Variables per facet:
  collaboration: ['collaboration_count', 'collaboration_pct']
  country: ['usage_count', 'usage_pct']
  onet_task: ['onet_task_count', 'onet_task_pct']
  request: ['request_count', 'request_pct']
  onet_task::collaboration: ['onet_task_collaboration_count', 'onet_task_collaboration_pct']
  request::collaboration: ['request_collaboration_count', 'request_collaboration_pct']
  state_us: ['usage_count', 'usage_pct']


---
## 2. Context: Indonesia's AUI

The AUI is the country-level RCA — Indonesia's share of global Claude usage divided by its share of global working-age population. This is our context number, not the focus of analysis.

In [8]:
# Country-level usage share
country_usage = (
    geo[(geo.facet == 'country') & (geo.variable == 'usage_pct')]
    [['geo_id', 'value']].rename(columns={'value': 'usage_pct'})
)

pop_clean = pop[['country_code', 'working_age_pop']].rename(columns={'country_code': 'geo_id'}).dropna()
pop_clean['pop_share'] = 100 * pop_clean.working_age_pop / pop_clean.working_age_pop.sum()

aui = country_usage.merge(pop_clean, on='geo_id', how='inner')
aui['AUI'] = aui.usage_pct / aui.pop_share

id_row = aui[aui.geo_id == INDONESIA].iloc[0]
median_aui = aui.AUI.median()
rank_usage = (aui.usage_pct > id_row.usage_pct).sum() + 1
rank_aui = (aui.AUI > id_row.AUI).sum() + 1

print('=== INDONESIA — CONTEXT (AUI is country-level RCA) ===\n')
print(f'  Share of global Claude.ai usage:  {id_row.usage_pct:.3f}%')
print(f'  Working-age population share:     {id_row.pop_share:.2f}%')
print(f'  AUI (usage share / pop share):    {id_row.AUI:.3f}')
print(f'  Global median AUI:                {median_aui:.3f}')
print(f'  Rank by absolute usage:           #{rank_usage} of {len(aui)}')
print(f'  Rank by AUI:                      #{rank_aui} of {len(aui)}')

=== INDONESIA — CONTEXT (AUI is country-level RCA) ===

  Share of global Claude.ai usage:  1.869%
  Working-age population share:     5.08%
  AUI (usage share / pop share):    0.368
  Global median AUI:                0.757
  Rank by absolute usage:           #11 of 165
  Rank by AUI:                      #113 of 165


---
## 3. Task- and request-level RCA profiles

Two views of Indonesia's AI adoption specialization:
- **SOC major group** — aggregated to broad occupational categories (22 groups)
- **Request clusters** — Anthropic's bottom-up taxonomy of what conversations are about

In [ ]:
# Discover the percentage variable names per facet
# (AEI uses facet-specific naming: 'onet_task_pct', 'request_pct', etc.)
task_pct_var = next(v for v in geo[geo.facet == 'onet_task'].variable.unique() if 'pct' in v.lower())
req_pct_var = next(v for v in geo[geo.facet == 'request'].variable.unique() if 'pct' in v.lower())
print(f'Task variable:    {task_pct_var}')
print(f'Request variable: {req_pct_var}')

task_usage = (
    geo[(geo.facet == 'onet_task') & (geo.variable == task_pct_var)]
    [['geo_id', 'cluster_name', 'value']]
    .rename(columns={'value': 'usage_pct', 'cluster_name': 'task'})
)
request_usage = (
    geo[(geo.facet == 'request') & (geo.variable == req_pct_var)]
    [['geo_id', 'cluster_name', 'value']]
    .rename(columns={'value': 'usage_pct', 'cluster_name': 'request'})
)

print(f'\nTask-country rows:     {len(task_usage):,}  ({task_usage.geo_id.nunique()} countries, {task_usage.task.nunique():,} tasks)')
print(f'Request-country rows:  {len(request_usage):,}  ({request_usage.geo_id.nunique()} countries, {request_usage.request.nunique():,} clusters)')

### 3.1 SOC major group RCA

Map tasks to SOC major groups via O\*NET task statements, then for each SOC group compute Indonesia's share vs. the global median share across countries.

In [ ]:
# Build task-to-SOC map
task_col = next((c for c in onet_tasks.columns if 'task' in c.lower() and onet_tasks[c].dtype == 'object'), None)
soc_col = next((c for c in onet_tasks.columns if 'onet' in c.lower() or 'soc' in c.lower()), None)

normalize = lambda s: str(s).strip().lower().rstrip('.')
onet_tasks['task_norm'] = onet_tasks[task_col].apply(normalize)
onet_tasks['soc_major'] = onet_tasks[soc_col].astype(str).str[:2]
onet_tasks['occ_code'] = onet_tasks[soc_col].astype(str).str.split('.').str[0]
task_to_occ = onet_tasks[['task_norm', 'soc_major', 'occ_code']].drop_duplicates('task_norm')

# Attach SOC info to task usage
task_usage['task_norm'] = task_usage.task.apply(normalize)
task_usage = task_usage.merge(task_to_occ, on='task_norm', how='left')

match_rate = task_usage.soc_major.notna().mean()
print(f'Task-to-SOC match rate: {match_rate:.0%}')

# For each country, aggregate to SOC major group share of that country's usage
soc_by_country = (
    task_usage.dropna(subset=['soc_major'])
    .groupby(['geo_id', 'soc_major'])['usage_pct']
    .sum()
    .reset_index()
)

# For each country, renormalize so SOC shares sum to 100 within mapped tasks
totals = soc_by_country.groupby('geo_id').usage_pct.sum().rename('total')
soc_by_country = soc_by_country.merge(totals, on='geo_id')
soc_by_country['share_pct'] = 100 * soc_by_country.usage_pct / soc_by_country.total

# Indonesia's profile
id_soc = soc_by_country[soc_by_country.geo_id == INDONESIA][['soc_major', 'share_pct']].rename(
    columns={'share_pct': 'Indonesia_%'}
)

# Global median and ASEAN average per SOC group
global_median = soc_by_country.groupby('soc_major').share_pct.median().rename('Global_median_%')
asean_avg = (
    soc_by_country[soc_by_country.geo_id.isin(ASEAN_PEERS)]
    .groupby('soc_major').share_pct.mean().rename('ASEAN_avg_%')
)

soc_table = id_soc.merge(global_median, on='soc_major').merge(asean_avg, on='soc_major', how='left')
soc_table['SOC_label'] = soc_table.soc_major.map(SOC_MAJOR_LABELS).fillna(soc_table.soc_major)

# RCA = Indonesia's share / Global median share
soc_table['RCA'] = soc_table['Indonesia_%'] / soc_table['Global_median_%'].replace(0, np.nan)
soc_table = soc_table.sort_values('RCA', ascending=False).reset_index(drop=True)

print('\n=== INDONESIA RCA BY SOC MAJOR GROUP ===\n')
display_cols = ['SOC_label', 'Indonesia_%', 'Global_median_%', 'ASEAN_avg_%', 'RCA']
print(soc_table[display_cols].to_string(index=False, float_format=lambda x: f'{x:7.2f}' if pd.notna(x) else '    NaN'))

soc_table.to_csv(OUTPUT_DIR / 'soc_rca.csv', index=False)

In [ ]:
# Visualize SOC RCA — show only groups with meaningful Indonesian presence
viz = soc_table[soc_table['Indonesia_%'] > 0.5].copy().sort_values('RCA')

fig, ax = plt.subplots(figsize=(12, max(5, 0.5 * len(viz))))
colors = ['#D85A30' if r > 1 else '#534AB7' for r in viz.RCA]
ax.barh(viz.SOC_label, viz.RCA, color=colors, edgecolor='white')
ax.axvline(1.0, color='#444', linestyle='--', linewidth=1.2, label='RCA = 1 (proportional)')
ax.set_xlabel('RCA (Indonesia share / Global median share)')
ax.set_title('Indonesia\'s revealed comparative advantage by SOC major group\n'
             'Orange = Indonesia over-specializes (RCA > 1); Purple = under-specializes (RCA < 1)')
ax.legend(loc='lower right')

for i, (label, rca) in enumerate(zip(viz.SOC_label, viz.RCA)):
    ax.text(rca + 0.05, i, f'{rca:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'soc_rca.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Request cluster RCA

The `request` facet is Anthropic's bottom-up taxonomy of conversation topics (not mapped through O\*NET). Useful for surfacing distinctive use cases like religious text translation or coursework assistance.

In [ ]:
# Global median per request cluster (across countries that have it)
global_req_median = (
    request_usage.groupby('request')
    .agg(global_median_pct=('usage_pct', 'median'),
         n_countries=('geo_id', 'nunique'))
    .reset_index()
)

id_req = request_usage[request_usage.geo_id == INDONESIA][['request', 'usage_pct']].rename(
    columns={'usage_pct': 'indonesia_pct'}
)

req_rca = id_req.merge(global_req_median, on='request', how='inner')
req_rca = req_rca[req_rca.n_countries >= 5].copy()  # require a reasonable baseline
req_rca['RCA'] = req_rca.indonesia_pct / req_rca.global_median_pct.replace(0, np.nan)

print(f'Request clusters analyzed: {len(req_rca)} (require >=5 countries for global baseline)')

# Filter to meaningful Indonesian presence before ranking
meaningful = req_rca[req_rca.indonesia_pct >= 0.5].copy()

print('\n=== TOP 15 OVER-SPECIALIZED (RCA > 1) ===\n')
for _, r in meaningful.nlargest(15, 'RCA').iterrows():
    print(f'  RCA={r.RCA:5.2f}x  ID={r.indonesia_pct:5.2f}%  median={r.global_median_pct:5.2f}%  {r.request[:75]}')

print('\n=== TOP 15 UNDER-SPECIALIZED (RCA < 1, meaningful global activity) ===\n')
for _, r in meaningful[meaningful.global_median_pct >= 1.0].nsmallest(15, 'RCA').iterrows():
    print(f'  RCA={r.RCA:5.2f}x  ID={r.indonesia_pct:5.2f}%  median={r.global_median_pct:5.2f}%  {r.request[:75]}')

req_rca.to_csv(OUTPUT_DIR / 'request_rca.csv', index=False)

In [ ]:
# Visualize top-15 over- and under-specialized request clusters
top_over = meaningful.nlargest(10, 'RCA')
top_under = meaningful[meaningful.global_median_pct >= 1.0].nsmallest(10, 'RCA')
combined = pd.concat([top_under, top_over]).sort_values('RCA')
combined['short'] = combined.request.apply(lambda s: s if len(s) < 65 else s[:62] + '...')

fig, ax = plt.subplots(figsize=(13, 8))
colors = ['#534AB7' if r < 1 else '#D85A30' for r in combined.RCA]
ax.barh(range(len(combined)), np.log2(combined.RCA), color=colors, edgecolor='white')
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(combined.short, fontsize=9)
ax.axvline(0, color='#444', linewidth=0.8)
ax.set_xlabel('log$_2$(RCA) — 0 means proportional, +1 means 2x global, -1 means 0.5x global')
ax.set_title('Indonesia\'s most- and least-specialized request clusters\n'
             'Orange = Indonesia over-specializes, Purple = under-specializes')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'request_rca.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
over = meaningful.nlargest(10, 'RCA')
under = meaningful[meaningful.global_median_pct >= 1.0].nsmallest(10, 'RCA')
combined = pd.concat([under, over]).sort_values('RCA').reset_index(drop=True)
combined['log2_RCA'] = np.log2(combined.RCA)
combined['short'] = combined.request.apply(lambda s: s if len(s) < 55 else s[:52] + '...')

fig, ax = plt.subplots(figsize=(12, max(6, 0.45 * len(combined))))
colors = ['#534AB7' if v < 0 else '#D85A30' for v in combined.log2_RCA]
ax.barh(range(len(combined)), combined.log2_RCA, color=colors, edgecolor='white', height=0.75)
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(combined.short, fontsize=9)

# Annotate each bar with the raw RCA value
for i, r in combined.iterrows():
    x_pos = r.log2_RCA + (0.1 if r.log2_RCA > 0 else -0.1)
    ha = 'left' if r.log2_RCA > 0 else 'right'
    ax.text(x_pos, i, f'{r.RCA:.1f}x ({r.indonesia_pct:.1f}% vs {r.global_median_pct:.1f}%)',
            va='center', ha=ha, fontsize=8, color='#333')

ax.set_xlabel('log$_2$(RCA)  —  +1 means 2x over-specialized, -1 means half the global rate')
ax.set_title('Indonesia\'s request cluster specialization')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'request_rca_log2.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Occupation-level RCA vs. US observed exposure

 We compute Indonesia's RCA at the occupation level (aggregating tasks to SOC codes) and plot it against Massenkoff & McCrory's US observed exposure. The Spearman correlation tests whether Indonesia's AI integration pattern follows the same occupational hierarchy as US-measured AI exposure.

In [ ]:
# Aggregate task usage to occupation level, per country, then renormalize so occupation shares sum to 100 within mapped tasks
occ_by_country = (
    task_usage.dropna(subset=['occ_code'])
    .groupby(['geo_id', 'occ_code'])['usage_pct']
    .sum()
    .reset_index()
)
totals = occ_by_country.groupby('geo_id').usage_pct.sum().rename('total')
occ_by_country = occ_by_country.merge(totals, on='geo_id')
occ_by_country['share_pct'] = 100 * occ_by_country.usage_pct / occ_by_country.total

# Indonesia's occupational shares
id_occ = occ_by_country[occ_by_country.geo_id == INDONESIA][['occ_code', 'share_pct']].rename(
    columns={'share_pct': 'indonesia_share'}
)

# Global median share per occupation (across countries that have it)
global_occ_stats = (
    occ_by_country.groupby('occ_code')
    .agg(global_median=('share_pct', 'median'),
         n_countries=('geo_id', 'nunique'))
    .reset_index()
)

# Build the analysis dataset: Indonesia share, global median, RCA, US exposure
df = (
    job_exposure
    .merge(id_occ, on='occ_code', how='left')
    .merge(global_occ_stats, on='occ_code', how='left')
)
df['indonesia_share'] = df.indonesia_share.fillna(0)
df['RCA'] = df.indonesia_share / df.global_median.replace(0, np.nan)

# For the Spearman test we need both axes defined and meaningful
both = df.dropna(subset=['RCA']).copy()
both = both[both.observed_exposure > 0]  # restrict to occupations with positive exposure
both = both[both.n_countries >= 5]        # require a reasonable baseline

print(f'Total US occupations in exposure data:  {len(job_exposure)}')
print(f'Occupations with Indonesian AI usage:   {(df.indonesia_share > 0).sum()}')
print(f'Occupations in analysis sample:         {len(both)}')
print(f'   (nonzero exposure, RCA defined, global baseline >=5 countries)')

In [ ]:
from scipy.stats import skew, kurtosis, shapiro

def describe(x, name):
    W, p = shapiro(x)
    return {
        'variable': name,
        'n': len(x),
        'mean_val': x.mean(),        # renamed to avoid collision
        'median_val': np.median(x),  # renamed to avoid collision
        'skewness': skew(x),
        'excess_kurtosis': kurtosis(x),
        'shapiro_W': W,
        'shapiro_p': p,
        'reject_normal': p < 0.05,
    }

diag = pd.DataFrame([
    describe(both.observed_exposure.values, 'US observed exposure'),
    describe(both.RCA.values, 'Indonesia RCA'),
])

print('=== DISTRIBUTIONAL DIAGNOSTIC ===\n')
for _, r in diag.iterrows():
    print(f'{r.variable}:')
    print(f'  n = {r.n}, mean = {r.mean_val:.4f}, median = {r.median_val:.4f}')
    print(f'  skewness = {r.skewness:+.3f}  (symmetric if near 0)')
    print(f'  excess kurtosis = {r.excess_kurtosis:+.3f}  (normal tails if near 0)')
    print(f'  Shapiro-Wilk W = {r.shapiro_W:.4f}, p = {r.shapiro_p:.2e}')
    verdict = 'REJECT normality' if r.reject_normal else 'Cannot reject normality'
    print(f'  -> {verdict} at alpha = 0.05\n')

diag.to_csv(OUTPUT_DIR / 'distributional_diagnostic.csv', index=False)

In [ ]:
# Spearman rank correlation matrix
from scipy.stats import spearmanr

corr_vars = both[['observed_exposure', 'RCA', 'indonesia_share', 'global_median']].copy()
corr_vars.columns = ['US exposure', 'Indonesia RCA', 'Indonesia share', 'Global median']

# Compute correlation matrix and p-value matrix
rho, pval = spearmanr(corr_vars.to_numpy())
rho = pd.DataFrame(rho, index=corr_vars.columns, columns=corr_vars.columns)
pval = pd.DataFrame(pval, index=corr_vars.columns, columns=corr_vars.columns)

# Format with significance stars
def fmt(r, p):
    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    return f'{r:+.3f}{stars}'

table = pd.DataFrame({
    col: [fmt(rho.loc[row, col], pval.loc[row, col]) if row != col else '1.000'
          for row in rho.index]
    for col in rho.columns
}, index=rho.index)

print('=== SPEARMAN CORRELATION MATRIX ===')
print(f'n = {len(both)}, * p<0.05, ** p<0.01, *** p<0.001\n')
print(table.to_string())

# Save and extract headline
table.to_csv(OUTPUT_DIR / 'spearman_matrix.csv')
sp = rho.loc['US exposure', 'Indonesia RCA']
sp_p = pval.loc['US exposure', 'Indonesia RCA']
print(f'\n Spearman rho = {sp:+.3f}, p = {sp_p:.4f}')

In [ ]:
# The main scatter plot
fig, ax = plt.subplots(figsize=(13, 8))

# Quadrants: RCA = 1 (specialization boundary), exposure median
exp_med = both.observed_exposure.median()

def quadrant(row):
    hi_exp = row.observed_exposure >= exp_med
    hi_rca = row.RCA >= 1
    return ('HH' if hi_exp and hi_rca
            else 'HL' if hi_exp and not hi_rca
            else 'LH' if hi_rca
            else 'LL')

both['quad'] = both.apply(quadrant, axis=1)
colors = {'HH': '#D85A30', 'HL': '#534AB7', 'LH': '#1D9E75', 'LL': '#B4B2A9'}
labels = {
    'HH': f'High exposure + over-specialized (n={(both.quad=="HH").sum()})',
    'HL': f'High exposure, under-specialized (n={(both.quad=="HL").sum()})',
    'LH': f'Low exposure, over-specialized (n={(both.quad=="LH").sum()})',
    'LL': f'Low / low (n={(both.quad=="LL").sum()})',
}

for q, c in colors.items():
    sub = both[both.quad == q]
    ax.scatter(sub.observed_exposure, sub.RCA, s=65, alpha=0.65,
               color=c, edgecolor='white', linewidth=0.5, label=labels[q])

# Annotate notable HH and HL occupations
for _, r in both[both.quad == 'HH'].nlargest(6, 'RCA').iterrows():
    short = r.title if len(str(r.title)) < 35 else r.title[:32] + '...'
    ax.annotate(short, (r.observed_exposure, r.RCA), xytext=(5, 3),
                textcoords='offset points', fontsize=8, color='#444')

ax.axhline(1.0, color='#888', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(exp_med, color='#888', linestyle='--', alpha=0.5, linewidth=1)
ax.set_yscale('log')
ax.set_xlabel('US observed exposure (Massenkoff & McCrory 2026)')
ax.set_ylabel('Indonesia RCA at occupation level (log scale)')
ax.set_title(f'Indonesia\'s AI adoption specialization vs. US-measured exposure\n'
             )
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rca_vs_exposure.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print the four quadrants so they can be discussed by name in the thesis
for q, description in [
    ('HH', 'HIGH EXPOSURE + OVER-SPECIALIZED (RCA > 1)\n  The observable adjustment cohort — Indonesia already on the curve'),
    ('HL', 'HIGH EXPOSURE, UNDER-SPECIALIZED (RCA < 1)\n  Latent adoption gap — AI capable of this work, Indonesia not using it yet'),
    ('LH', 'LOW EXPOSURE, OVER-SPECIALIZED (RCA > 1)\n  Augmentation-only zone — Indonesia uses AI in areas with low displacement risk'),
]:
    sub = both[both.quad == q].sort_values('RCA', ascending=False)
    print(f'\n{"=" * 70}')
    print(f'{description}')
    print(f'n = {len(sub)}')
    print('=' * 70)
    if len(sub):
        top = sub.head(12)[['occ_code', 'title', 'observed_exposure', 'RCA']].copy()
        top['observed_exposure'] = top.observed_exposure.apply(lambda x: f'{x:.3f}')
        top['RCA'] = top.RCA.apply(lambda x: f'{x:.2f}')
        print(top.to_string(index=False))

# LL quadrant: reported as summary (face-validity check)
ll = both[both.quad == 'LL'].copy()
ll['soc_major'] = ll.occ_code.str[:2]
print(f'\n{"=" * 70}')
print('LOW EXPOSURE + UNDER-SPECIALIZED (RCA < 1)')
print('  Face-validity check — both signals agree AI is not in this occupation')
print(f'n = {len(ll)}')
print('=' * 70)

if len(ll):
    print('\nTop SOC major groups in LL quadrant:')
    top_socs = ll.soc_major.map(SOC_MAJOR_LABELS).fillna(ll.soc_major).value_counts().head(6)
    for soc, count in top_socs.items():
        print(f'  {soc:40s} {count} occupations')

    print('\nIllustrative examples (random sample of 8):')
    sample = ll.sample(min(8, len(ll)), random_state=42)
    for _, r in sample.iterrows():
        print(f'  {r.occ_code}  {str(r.title)[:55]:55s}  exposure={r.observed_exposure:.3f}, RCA={r.RCA:.2f}')

both.to_csv(OUTPUT_DIR / 'occupation_rca_vs_exposure.csv', index=False)

---
## 5. Summary

In [ ]:
summary = f'''
INDONESIA'S REVEALED COMPARATIVE ADVANTAGE IN AI ADOPTION
==========================================================

CONTEXT
  Indonesia AUI:              {id_row.AUI:.3f}
  Global median AUI:          {median_aui:.3f}
  Rank by usage:              #{rank_usage} of {len(aui)}
  Rank by AUI:                #{rank_aui} of {len(aui)}

SOC MAJOR GROUP RCA (top 3)
'''
for _, r in soc_table.head(3).iterrows():
    summary += f'  {r.SOC_label:35s} RCA = {r.RCA:5.2f}  ({r["Indonesia_%"]:5.2f}% ID vs {r["Global_median_%"]:5.2f}% median)\n'

summary += '\nSOC MAJOR GROUP RCA (bottom 3, with Indonesian presence)\n'
for _, r in soc_table[soc_table['Indonesia_%'] > 0.5].tail(3).iterrows():
    summary += f'  {r.SOC_label:35s} RCA = {r.RCA:5.2f}  ({r["Indonesia_%"]:5.2f}% ID vs {r["Global_median_%"]:5.2f}% median)\n'

summary += f'''
REQUEST CLUSTER — MOST DISTINCTIVELY INDONESIAN
  {meaningful.nlargest(1, 'RCA').iloc[0].request[:80]}
  RCA = {meaningful.nlargest(1, 'RCA').iloc[0].RCA:.1f}x global

HEADLINE — OCCUPATION-LEVEL RCA vs. US OBSERVED EXPOSURE
  Spearman rho:    {sp:+.3f}   (p = {sp_p:.4f})
  Sample size:     n = {len(both)} occupations
  HH quadrant:     {(both.quad=="HH").sum()} occupations (high US exposure + Indonesia over-specialized)
  HL quadrant:     {(both.quad=="HL").sum()} occupations (high US exposure, under-specialized)

FILES
  outputs/soc_rca.csv
  outputs/request_rca.csv
  outputs/occupation_rca_vs_exposure.csv
  outputs/soc_rca.png
  outputs/request_rca.png
  outputs/rca_vs_exposure.png
'''
print(summary)

with open(OUTPUT_DIR / 'rca_findings.txt', 'w') as f:
    f.write(summary)

---
